# GD-CLASS Explorer v5 — Late Glass Model

**Date:** March 13, 2026 | **Status:** CURRENT

---

### What this notebook shows

Tom proposed inverting the κ(z) timeline ("Late Glass"):
- **Before recombination (z > 1100):** κ = 1.0 — standard gravity, sound horizon r_s intact
- **At recombination (z = 1100):** glass transition freezes κ at κ_c
- **After recombination (z < 1100):** κ decays from κ_c back to 1.0 via stretched exponential

The motivation: protect r_s (which Planck measures precisely) while modifying the angular diameter distance D_A to raise H₀.

**The result:** The χ² wall does not drop. At every H₀, the best κ is 1.000 (standard ΛCDM).

### Previous notebooks (all preserved)
- **v1** — Rigid inclusion model (Feb 19)
- **v2** — Compliant inclusion model (Feb 26)
- **v3** — Parameter-fitted background model (Mar 5)
- **v4** — MCMC results: κ = 0.998 (Mar 12)

In [ ]:
# @title Setup — Run this cell first { display-mode: "form" }
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 13,
    'axes.labelsize': 14,
    'axes.titlesize': 15,
    'legend.fontsize': 11,
    'figure.dpi': 120,
})

# ── Late Glass 2D Grid Scan Results ──
# κ = 1.0 before z=1100, κ_c after, decaying to 1.0 by z=0
# 8 κ_c values × 7 h values = 56 CLASS runs
# Other params fixed at Planck 2018 best-fit

# Columns: kappa_c, chi2/dof for each H0
h_values = [0.6736, 0.68, 0.69, 0.70, 0.71, 0.72, 0.73]
H0_values = [h * 100 for h in h_values]

kappa_values = [0.85, 0.90, 0.94, 0.96, 0.98, 0.99, 0.995, 1.0]
chi2_grid = np.array([
    [7.527, 7.862, 8.390, 8.920, 9.452, 9.983, 10.512],   # 0.85
    [4.216, 4.497, 4.952, 5.422, 5.905, 6.398, 6.901],    # 0.90
    [2.338, 2.537, 2.872, 3.236, 3.623, 4.034, 4.464],    # 0.94
    [1.712, 1.857, 2.113, 2.404, 2.726, 3.076, 3.451],    # 0.96
    [1.318, 1.402, 1.569, 1.775, 2.020, 2.298, 2.607],    # 0.98
    [1.214, 1.265, 1.383, 1.543, 1.744, 1.982, 2.254],    # 0.99
    [1.186, 1.220, 1.312, 1.448, 1.627, 1.844, 2.096],    # 0.995
    [1.174, 1.190, 1.256, 1.369, 1.524, 1.719, 1.951],    # 1.000
])

chi2_lcdm = 1.1736  # ΛCDM baseline
dof = 2472

# 1D scan (fixed h=0.6736)
scan_1d = {
    'kappa':    [0.85, 0.90, 0.92, 0.94, 0.96, 0.97, 0.98, 0.99, 0.995, 0.999, 1.0],
    'chi2_dof': [7.527, 4.216, 3.180, 2.338, 1.712, 1.485, 1.318, 1.214, 1.186, 1.175, 1.174],
    'delta_chi2': [15705, 7521, 4959, 2879, 1330, 769, 357, 99, 29, 2.7, 0.0],
}

# MCMC result (from v4, for reference)
mcmc_kappa = 0.998
mcmc_H0 = 66.8

print('Data loaded: Late Glass 2D grid scan (56 CLASS runs) + 1D scan (11 points)')

## 1. The κ(z) Timeline — What Late Glass Does

**What this chart shows:** The shape of κ as a function of redshift z for three different κ_c values. This is Tom's proposed "Late Glass" timeline.

**How to read it:**
- The x-axis is redshift z (time runs right-to-left: right side is the early universe, left side is today).
- Before recombination (z > 1100): κ = 1.0 for ALL models. Standard gravity. Sound horizon protected.
- At z = 1100: the glass transition happens. κ drops to κ_c.
- After recombination: κ slowly relaxes back to 1.0 via stretched exponential decay.
- The deeper the dip (lower κ_c), the stronger the temporary gravity boost.

**The physical picture:** Like Venetian glassblowing — the vacuum is molten (κ=1) in the hot early universe, freezes into glass (κ=κ_c) when the photons escape at recombination, then slowly softens back to liquid (κ=1) as the universe expands.

**Why this matters for H₀:** The integral of 1/H(z) from z=0 to z=1089 gives the angular diameter distance D_A. Stronger gravity (lower κ) means larger H(z) in the denominator, shrinking D_A. Tom hoped this shrinkage could mimic a higher H₀.

In [ ]:
# @title Figure 1: Late Glass κ(z) profile { display-mode: "form" }

fig, ax = plt.subplots(figsize=(12, 5))

z = np.linspace(0, 2000, 2000)
z_onset = 1100
beta = 0.5

for kc, color, ls in [(0.85, 'C3', '-'), (0.94, 'C1', '--'), (0.99, 'C0', ':')]:
    kappa = np.ones_like(z)
    mask = (z < z_onset) & (z > 0)
    t = z[mask] / z_onset
    decay = np.exp(-((1.0 - t) / 0.5)**beta)
    kappa[mask] = 1.0 + (kc - 1.0) * decay
    ax.plot(z, kappa, color=color, ls=ls, lw=2.5,
            label=f'κ_c = {kc} (G_eff/G_N = {1/kc:.3f})')

ax.axvline(1100, color='gray', ls=':', lw=1, alpha=0.5)
ax.text(1110, 0.82, 'recombination\nz = 1100', fontsize=10, color='gray')

ax.annotate('κ = 1.0\n(standard gravity)', xy=(1500, 1.0), fontsize=11,
            ha='center', color='green',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

ax.annotate('κ = 1.0\n(relaxed today)', xy=(50, 1.0), fontsize=11,
            ha='center', color='green',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

ax.set_xlabel('Redshift z (→ past)')
ax.set_ylabel('κ(z)')
ax.set_title('Late Glass Model: κ(z) Timeline')
ax.set_xlim(0, 2000)
ax.set_ylim(0.80, 1.05)
ax.legend(loc='lower right', fontsize=11)
ax.invert_xaxis()  # past on right, today on left
plt.tight_layout()
plt.show()

print('Before z=1100: κ=1.0 (sound horizon protected)')
print('After z=1100: κ dips to κ_c then relaxes back to 1.0')
print('The entire zone from z=1100 to z=0 is modified — Planck photons travel through it.')

## 2. The 1D Grid Scan — χ² vs κ_c

**What this chart shows:** For each κ_c value, we ran CLASS with the Late Glass profile and computed χ² against Planck 2018 TT data (2,507 points, ℓ ≥ 30). All other cosmological parameters held at Planck best-fit.

**How to read it:**
- Left panel: χ²/dof for each κ_c. The horizontal green line is ΛCDM (1.174). Lower is better.
- Right panel: Δχ² (penalty compared to ΛCDM) on a log scale. The gray dashed line at Δχ² = 3.84 is the 95% exclusion threshold.
- At κ_c = 0.85 (Tom's prediction): χ²/dof = 7.5, Δχ² = 15,700. Catastrophically excluded.
- At κ_c = 0.99: χ²/dof = 1.21, Δχ² = 99. Still strongly excluded.
- Only κ_c > 0.999 is marginally consistent with the data.

**What it means:** The Late Glass model has the same χ² wall as the Early Glass model. Protecting r_s was not enough — the ISW and lensing damage from the post-recombination κ modification is detected by Planck.

In [ ]:
# @title Figure 2: Late Glass 1D scan — χ² wall persists { display-mode: "form" }

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

kk = np.array(scan_1d['kappa'])
cc = np.array(scan_1d['chi2_dof'])
dd = np.array(scan_1d['delta_chi2'])

# Left: chi2/dof
ax1.plot(kk, cc, 'C3o-', ms=8, lw=2)
ax1.axhline(chi2_lcdm, color='green', ls=':', lw=1.5, label=f'ΛCDM: χ²/dof = {chi2_lcdm:.3f}')
ax1.axhline(1.3, color='orange', ls='--', lw=1, alpha=0.5, label='"publishable" threshold (1.3)')

for i in range(len(kk)):
    if cc[i] < 4:
        ax1.annotate(f'{cc[i]:.3f}', (kk[i], cc[i]),
                     textcoords='offset points', xytext=(5, 8), fontsize=8)

ax1.set_xlabel(r'$\kappa_c$')
ax1.set_ylabel(r'$\chi^2$/dof')
ax1.set_title('Late Glass: CMB Fit Quality')
ax1.legend(fontsize=10)
ax1.set_xlim(0.84, 1.01)
ax1.set_ylim(1.0, 3.0)

# Right: delta chi2 (log)
mask_pos = dd > 0
ax2.semilogy(kk[mask_pos], dd[mask_pos], 'C3o-', ms=8, lw=2)
ax2.axhline(3.84, color='gray', ls='--', lw=1, label='95% CL (Δχ² = 3.84)')

for i in range(len(kk)):
    if dd[i] > 0:
        ax2.annotate(f'κ={kk[i]:.3f}: Δχ²={dd[i]:.0f}',
                     (kk[i], dd[i]), textcoords='offset points',
                     xytext=(8, 3), fontsize=8)

ax2.set_xlabel(r'$\kappa_c$')
ax2.set_ylabel(r'$\Delta\chi^2$ vs ΛCDM')
ax2.set_title('Late Glass: χ² Penalty (log scale)')
ax2.legend(fontsize=10)
ax2.set_xlim(0.84, 1.01)

plt.tight_layout()
plt.show()

print(f'κ_c = 0.85 (Tom\'s prediction): Δχ² = {15705:.0f} — excluded beyond any doubt')
print(f'κ_c = 0.99: Δχ² = 99 — still strongly excluded')
print(f'κ_c = 0.999: Δχ² = 2.7 — marginally allowed')

## 3. The 2D Grid — κ_c × H₀

**What this chart shows:** A heatmap of χ²/dof for 56 combinations of κ_c and H₀. This is the key test — can the Late Glass model find ANY combination of κ and H₀ that fits Planck well AND has H₀ > 70?

**How to read it:**
- Each cell shows χ²/dof. Blue = good fit. Red = bad fit.
- The best fit (darkest blue) is always in the bottom-right corner: κ_c = 1.0, H₀ = 67.4 — standard ΛCDM.
- Moving left (lower κ_c): fit gets worse at every H₀.
- Moving up (higher H₀): fit gets worse at every κ_c.
- There is no blue cell anywhere near H₀ = 73.

**The critical finding:** At every H₀, the best κ_c is 1.000. The Late Glass modification never helps — it only adds damage. There is no "geometric degeneracy" to exploit.

**Why not?** Tom expected that shrinking D_A (via stronger post-recombination gravity) could trade off with H₀. But the CMB encodes more than just θ = r_s/D_A. The ISW effect and gravitational lensing independently constrain the post-recombination expansion history, breaking the degeneracy.

In [ ]:
# @title Figure 3: 2D heatmap — κ_c × H₀ { display-mode: "form" }

fig, ax = plt.subplots(figsize=(12, 7))

# Clip for better color range
chi2_display = np.clip(chi2_grid, 1.0, 4.0)

im = ax.imshow(chi2_display, aspect='auto', cmap='RdYlBu_r',
               vmin=1.1, vmax=4.0, origin='lower')

# Labels
ax.set_xticks(range(len(H0_values)))
ax.set_xticklabels([f'{h:.1f}' for h in H0_values])
ax.set_yticks(range(len(kappa_values)))
ax.set_yticklabels([f'{k:.3f}' for k in kappa_values])
ax.set_xlabel('H₀ (km/s/Mpc)', fontsize=14)
ax.set_ylabel('κ_c', fontsize=14)
ax.set_title('Late Glass: χ²/dof for each (κ_c, H₀) combination', fontsize=15)

# Annotate each cell
for i in range(len(kappa_values)):
    for j in range(len(H0_values)):
        val = chi2_grid[i, j]
        color = 'white' if val > 2.5 else 'black'
        marker = ''
        if val < 1.2:
            marker = ' *'
        elif val < 1.3:
            marker = ' .'
        ax.text(j, i, f'{val:.2f}{marker}', ha='center', va='center',
                fontsize=9, color=color, fontweight='bold')

cbar = plt.colorbar(im, ax=ax, label='χ²/dof')

# Mark ΛCDM best
ax.plot(0, 7, 'g*', ms=20, markeredgecolor='black', markeredgewidth=1.5)
ax.annotate('ΛCDM\nbest fit', xy=(0, 7), xytext=(1.5, 6.5),
            fontsize=11, color='green', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='green', lw=2))

plt.tight_layout()
plt.show()

print('* = publishable (χ²/dof < 1.2)')
print('. = marginal (χ²/dof < 1.3)')
print()
print('At EVERY H₀, the best κ_c is 1.000 (standard ΛCDM).')
print('Late Glass never improves the fit. The χ² wall persists.')

## 4. Best Fit at Each H₀ — The Bottom Line

**What this chart shows:** For each target H₀, we pick the κ_c that gives the best χ². This is the most favorable case for the Late Glass model — the optimal κ at each H₀.

**How to read it:**
- Blue bars show χ²/dof. The green dashed line is ΛCDM (1.174).
- Red dashed line at χ²/dof = 1.3 marks the rough "publishable" threshold.
- The orange bar labels show the optimal κ_c at each H₀.

**The result:** At every H₀, the optimal κ_c is 1.000 (ΛCDM). The Late Glass model provides zero improvement. And the χ² wall rises steeply: by H₀ = 70, χ²/dof = 1.37 (excluded). By H₀ = 73, χ²/dof = 1.95 (catastrophically excluded).

In [ ]:
# @title Figure 4: Best fit at each H₀ { display-mode: "form" }

fig, ax = plt.subplots(figsize=(10, 6))

# Find best kappa for each H0
best_chi2 = []
best_kappa = []
for j in range(len(H0_values)):
    col = chi2_grid[:, j]
    best_idx = np.argmin(col)
    best_chi2.append(col[best_idx])
    best_kappa.append(kappa_values[best_idx])

colors = ['green' if c < 1.2 else 'orange' if c < 1.3 else 'red' for c in best_chi2]
bars = ax.bar(range(len(H0_values)), best_chi2, color=colors, alpha=0.7, edgecolor='black')

# Labels
for i, (c, k) in enumerate(zip(best_chi2, best_kappa)):
    ax.text(i, c + 0.02, f'κ={k:.3f}\n{c:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.axhline(chi2_lcdm, color='green', ls='--', lw=1.5, label=f'ΛCDM: {chi2_lcdm:.3f}')
ax.axhline(1.3, color='red', ls='--', lw=1, alpha=0.5, label='Publishable threshold (1.3)')

ax.set_xticks(range(len(H0_values)))
ax.set_xticklabels([f'{h:.1f}' for h in H0_values])
ax.set_xlabel('H₀ (km/s/Mpc)', fontsize=14)
ax.set_ylabel('Best χ²/dof (optimized over κ_c)', fontsize=13)
ax.set_title('Late Glass: Best Possible Fit at Each H₀', fontsize=15)
ax.legend(fontsize=11)
ax.set_ylim(1.1, 2.1)

plt.tight_layout()
plt.show()

print('At every H₀, the best κ_c is 1.000 — Late Glass provides zero improvement over ΛCDM.')
print(f'H₀ = 73.0 → χ²/dof = {best_chi2[-1]:.3f} (excluded)')
print(f'H₀ = 67.4 → χ²/dof = {best_chi2[0]:.3f} (= ΛCDM)')

## 5. Interactive Explorer — Late Glass Edition

**What this does:** Slide κ_c and H₀ to see the χ²/dof for any Late Glass configuration.

**The key insight from sliding:** No matter what κ_c you choose, increasing H₀ always makes the fit worse. And at any fixed H₀, decreasing κ_c (stronger gravity) always makes it worse. There is no sweet spot.

In [ ]:
# @title Interactive: Slide κ_c and H₀ { display-mode: "form" }

try:
    from ipywidgets import interact, FloatSlider
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print('(Widgets not available — showing static version)')

from scipy.interpolate import RegularGridInterpolator

# Build interpolator from grid
kk_arr = np.array(kappa_values)
hh_arr = np.array(H0_values)
interp = RegularGridInterpolator((kk_arr, hh_arr), chi2_grid, 
                                  method='linear', bounds_error=False, fill_value=None)

def plot_late_glass(kappa_c=0.98, H0=69.0):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Left: κ(z) profile
    z = np.linspace(0, 2000, 1000)
    kappa = np.ones_like(z)
    mask = z < 1100
    t = z[mask] / 1100
    decay = np.exp(-((1.0 - t) / 0.5)**0.5)
    kappa[mask] = 1.0 + (kappa_c - 1.0) * decay
    
    ax1.plot(z, kappa, 'C0-', lw=2.5)
    ax1.axvline(1100, color='gray', ls=':', alpha=0.5)
    ax1.axhline(1.0, color='green', ls=':', alpha=0.5)
    ax1.set_xlabel('Redshift z')
    ax1.set_ylabel('κ(z)')
    ax1.set_title(f'Late Glass profile: κ_c = {kappa_c:.3f}')
    ax1.set_xlim(0, 2000)
    ax1.set_ylim(0.80, 1.05)
    ax1.invert_xaxis()
    
    # Right: position on 2D grid
    chi2_display = np.clip(chi2_grid, 1.0, 4.0)
    im = ax2.imshow(chi2_display, aspect='auto', cmap='RdYlBu_r',
                    vmin=1.1, vmax=4.0, origin='lower',
                    extent=[H0_values[0]-0.5, H0_values[-1]+0.5,
                            kappa_values[0]-0.01, kappa_values[-1]+0.01])
    
    # Current point
    try:
        chi2_here = float(interp([[kappa_c, H0]]))
    except:
        chi2_here = float('nan')
    
    ax2.plot(H0, kappa_c, 'ko', ms=15, markeredgewidth=2)
    ax2.annotate(f'χ²/dof = {chi2_here:.3f}',
                 xy=(H0, kappa_c), xytext=(H0+1, kappa_c+0.02),
                 fontsize=12, fontweight='bold',
                 arrowprops=dict(arrowstyle='->', color='black'),
                 bbox=dict(boxstyle='round', facecolor='white'))
    
    ax2.set_xlabel('H₀ (km/s/Mpc)')
    ax2.set_ylabel('κ_c')
    ax2.set_title('Position on χ² grid')
    plt.colorbar(im, ax=ax2, label='χ²/dof')
    
    plt.tight_layout()
    plt.show()
    
    if chi2_here > 1.3:
        print(f'  EXCLUDED: χ²/dof = {chi2_here:.3f} (> 1.3)')
    elif chi2_here > 1.2:
        print(f'  MARGINAL: χ²/dof = {chi2_here:.3f}')
    else:
        print(f'  ALLOWED: χ²/dof = {chi2_here:.3f}')

if HAS_WIDGETS:
    interact(plot_late_glass,
             kappa_c=FloatSlider(value=0.98, min=0.85, max=1.0, step=0.01,
                                description='κ_c:', readout_format='.2f',
                                style={'description_width': '40px'},
                                layout={'width': '500px'}),
             H0=FloatSlider(value=69.0, min=67.0, max=73.0, step=0.5,
                            description='H₀:', readout_format='.1f',
                            style={'description_width': '40px'},
                            layout={'width': '500px'}))
else:
    plot_late_glass(0.98, 69.0)

## 6. Early Glass vs Late Glass — Both Directions Tested

**What this chart shows:** A side-by-side comparison of both glass models:
- **Early Glass** (v1-v4): κ_c applied before recombination, damages r_s directly
- **Late Glass** (v5): κ_c applied after recombination, damages ISW and lensing

**The conclusion:** Both directions converge on the same answer — κ must be very close to 1.0. The CMB constrains both the early universe (through acoustic peaks) and the late universe (through ISW + lensing). There is no era where a large κ deviation can hide from Planck.

In [ ]:
# @title Figure 6: Early Glass vs Late Glass — comparison { display-mode: "form" }

fig, ax = plt.subplots(figsize=(10, 6))

# Early Glass results (from MCMC / grid scan)
early_kappa = [1.000, 0.990, 0.970, 0.960, 0.950]
early_dchi2 = [0.0, 42.4, 348.2, 719.6, 1298.6]

# Late Glass results (from this scan)
late_kappa = [1.0, 0.999, 0.995, 0.99, 0.98, 0.97, 0.96, 0.94]
late_dchi2 = [0.0, 2.7, 29, 99, 357, 769, 1330, 2879]

ax.semilogy(early_kappa, [d + 1 for d in early_dchi2], 'C0o-', ms=8, lw=2,
            label='Early Glass (κ before recombination)')
ax.semilogy(late_kappa, [d + 1 for d in late_dchi2], 'C3s-', ms=8, lw=2,
            label='Late Glass (κ after recombination)')

ax.axhline(3.84 + 1, color='gray', ls='--', lw=1, label='95% CL threshold')

ax.set_xlabel(r'$\kappa_c$', fontsize=14)
ax.set_ylabel(r'$\Delta\chi^2$ + 1 (log scale)', fontsize=13)
ax.set_title('Both Directions Tested — Both Say κ ≈ 1.0', fontsize=15)
ax.legend(fontsize=11)
ax.set_xlim(0.93, 1.005)

plt.tight_layout()
plt.show()

print('Early Glass: damages r_s → acoustic peaks shift → χ² wall')
print('Late Glass: preserves r_s but damages ISW + lensing → χ² wall')
print('Both converge: κ must be > 0.996 at 95% CL')

---

## Summary

Tom proposed the Late Glass model to test whether inverting the κ(z) timeline could avoid the χ² wall.

**Implementation:** κ = 1.0 before recombination (protecting r_s), κ = κ_c after recombination (modifying D_A), decaying back to κ = 1.0 today.

**Result:** The χ² wall persists.

| | Early Glass | Late Glass |
|---|---|---|
| κ active | before z = 1100 | after z = 1100 |
| What breaks | r_s (acoustic peaks) | ISW + lensing |
| Best κ_c | 0.998 (MCMC) | 1.000 (grid scan) |
| H₀ achieved | 66.8 | 67.4 |
| Can reach 73? | No | No |

**Both directions now tested. Both say the same thing: the CMB data constrains κ to be very close to 1.0, and the Hubble tension cannot be resolved by modifying κ in either era.**

### All notebooks
| Notebook | Model | Date |
|---|---|---|
| v1 | Rigid inclusions (κ > 1) | Feb 19 |
| v2 | Compliant inclusions (κ < 1) | Feb 26 |
| v3 | Parameter-fitted background | Mar 5 |
| v4 | MCMC results (κ = 0.998) | Mar 12 |
| **v5** | **Late Glass (this notebook)** | **Mar 13** |